# Desire Paths: Two Models, One Rule Apart

We've now looked closely at the real NetLogo model this is based on, so this version is built with that understanding in mind:

- Like the NetLogo original, walkers here **always know exactly where their destination is** (an omniscient "compass," not something they discover by exploring). We're not modeling *that* kind of search here — see our earlier discussion for why relaxing it is a much bigger, separate model.
- What *does* vary between our two models is much narrower and more local: **when a walker has several equally-good steps available, does it choose based on the past, or not?**

That single difference is the entire experiment. Everything else — the world, the walkers, the destinations, which steps are even allowed — is identical in both models.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# --- The world: a grid, and three fixed destinations ("buildings") ---
WIDTH, HEIGHT = 41, 31
BUILDINGS = [(3, 5), (3, 25), (37, 15)]

print(f"A {WIDTH} x {HEIGHT} grid with {len(BUILDINGS)} buildings: {BUILDINGS}")


## Step 1 — Which moves are even allowed?

A walker only ever considers the up-to-8 neighboring cells that would take it **strictly closer** to its destination. This is the "compass" part: it always knows the straight-line distance to where it's going, and never deliberately steps backward.

This function just answers one question: *from here, which of my 8 neighboring cells would shrink the distance to my goal?*


In [ ]:
def closer_neighbors(x, y, goal_x, goal_y):
    """Return every neighboring cell that is strictly closer to
    (goal_x, goal_y) than (x, y) currently is.

    A walker will only ever step into one of the cells this returns.
    """
    current_distance = (x - goal_x) ** 2 + (y - goal_y) ** 2

    eight_directions = [
        (1, 0), (-1, 0), (0, 1), (0, -1),
        (1, 1), (1, -1), (-1, 1), (-1, -1),
    ]

    closer_cells = []
    for dx, dy in eight_directions:
        nx, ny = x + dx, y + dy
        if 0 <= nx < WIDTH and 0 <= ny < HEIGHT:
            new_distance = (nx - goal_x) ** 2 + (ny - goal_y) ** 2
            if new_distance < current_distance:
                closer_cells.append((nx, ny))

    return closer_cells

# quick sanity check: standing at (10, 10), heading to building (37, 15)
print("Example -- cells that get you closer:", closer_neighbors(10, 10, 37, 15))


## Step 2 — The ONE rule that differs between our two models

Usually a walker has 2 or 3 equally "closer" cells to choose from (diagonal moves and straight moves are often both valid). **How it breaks that tie is the entire experiment:**

- **Model A — No memory.** Pick uniformly at random among the closer cells. The walker has no idea, and doesn't care, whether anyone has walked here before.
- **Model B — Memory + feedback.** Most of the time, pick whichever closer cell has been stepped on the most so far (ties broken randomly). The rest of the time (`1 - follow_probability`), still pick randomly — so exploration never fully stops, and the walker isn't rigidly locked onto whatever became popular first.

Read this function once, carefully — it's the heart of the whole notebook.


In [ ]:
def choose_next_cell(candidates, popularity, use_memory, follow_probability, rng):
    """Decide which of the candidate cells to step into next.

    use_memory=False (Model A): always choose randomly.
    use_memory=True  (Model B): usually choose the most-popular candidate,
                                 but still choose randomly sometimes.
    """
    # Model A, OR Model B's random "still exploring" branch:
    if not use_memory or rng.random() > follow_probability:
        return candidates[rng.integers(len(candidates))]

    # Model B's "follow the crowd" branch:
    popularity_values = [popularity[cy, cx] for cx, cy in candidates]
    best_value = max(popularity_values)
    most_popular_cells = [
        cell for cell, value in zip(candidates, popularity_values)
        if value == best_value
    ]
    return most_popular_cells[rng.integers(len(most_popular_cells))]


## Step 3 — Put it together: walkers shuttling between buildings

Each walker repeatedly walks from one building to another, step by step, using `closer_neighbors` to see its options and `choose_next_cell` to pick one. Every cell a walker steps on gets its `popularity` count increased by 1 — that's the only thing either model tracks about the past.


In [ ]:
def run_simulation(use_memory, seed, n_walkers=80, total_steps=20000, follow_probability=0.85):
    """Simulate `n_walkers` shuttling between buildings for a combined
    `total_steps` footsteps. Returns a HEIGHT x WIDTH grid counting how
    many times each cell was stepped on.
    """
    rng = np.random.default_rng(seed)
    popularity = np.zeros((HEIGHT, WIDTH), dtype=int)

    # every walker starts at one building, heading to a different one
    positions, goals = [], []
    for _ in range(n_walkers):
        start_idx, goal_idx = rng.choice(len(BUILDINGS), size=2, replace=False)
        positions.append(BUILDINGS[start_idx])
        goals.append(BUILDINGS[goal_idx])

    steps_taken = 0
    while steps_taken < total_steps:
        for i in range(n_walkers):
            x, y = positions[i]
            goal_x, goal_y = goals[i]

            # arrived: head to a different building next
            if (x, y) == (goal_x, goal_y):
                other_buildings = [b for b in BUILDINGS if b != (goal_x, goal_y)]
                goals[i] = other_buildings[rng.integers(len(other_buildings))]
                continue

            candidates = closer_neighbors(x, y, goal_x, goal_y)
            next_cell = choose_next_cell(candidates, popularity, use_memory, follow_probability, rng)

            positions[i] = next_cell
            popularity[next_cell[1], next_cell[0]] += 1
            steps_taken += 1
            if steps_taken >= total_steps:
                break

    return popularity

print("Simulation function ready.")


## Step 4 — Run both models once, and look at them

Same seed, same walker count, same step budget. The only thing that changes is `use_memory`.


In [ ]:
model_A = run_simulation(use_memory=False, seed=123)   # no memory
model_B = run_simulation(use_memory=True, seed=123)     # memory + feedback

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, popularity, title in [
    (axes[0], model_A, "Model A -- No memory"),
    (axes[1], model_B, "Model B -- Memory + feedback"),
]:
    ax.imshow(popularity, origin="lower", cmap="inferno")
    bx, by = zip(*BUILDINGS)
    ax.scatter(bx, by, marker="s", s=80, c="cyan", edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
plt.suptitle("Cumulative foot traffic (brighter = more steps taken on that cell)")
plt.tight_layout()
plt.show()


## Step 5 — Summary statistics for the two models

A picture tells you *that* something is different. These numbers tell you **how**:

- **cells ever used** — how much of the map saw any traffic at all.
- **busiest cell's traffic** — the single most-walked-on cell's footstep count.
- **top-5% share** — of all footsteps taken, what fraction landed in the busiest 5% of used cells? Higher = more concentrated onto a few preferred routes.
- **traffic entropy** — a single number measuring how spread out the traffic is (higher = more spread out, lower = more concentrated). This is the same idea as top-5% share, computed a different way, as a cross-check.


In [ ]:
def summarize(popularity, model_name):
    """Compute a few simple numbers describing how concentrated
    the foot traffic became."""
    used_cells = popularity[popularity > 0]
    total_footsteps = used_cells.sum()

    sorted_traffic = np.sort(used_cells)[::-1]
    n_top = max(1, int(len(sorted_traffic) * 0.05))
    top5_share = sorted_traffic[:n_top].sum() / total_footsteps

    p = used_cells / total_footsteps
    traffic_entropy = -(p * np.log(p)).sum()

    return {
        "model": model_name,
        "total footsteps": int(total_footsteps),
        "cells ever used": int(len(used_cells)),
        "busiest cell's traffic": int(used_cells.max()),
        "top-5% share": round(top5_share, 3),
        "traffic entropy": round(traffic_entropy, 3),
    }

summary_table = pd.DataFrame([
    summarize(model_A, "A: No memory"),
    summarize(model_B, "B: Memory + feedback"),
]).set_index("model")

summary_table


**What to read off this table:** Model B concentrates a noticeably larger share of all footsteps onto a small set of cells, and has lower traffic entropy (more spread-out traffic has *higher* entropy) — both numbers agree that memory makes movement more organized around a few preferred routes, even though it doesn't use noticeably more or fewer cells overall.


## Step 6 — The real test: does this repeat?

One run each isn't enough to know *what kind* of difference this is. Let's run both models 5 times, changing only the random seed, and summarize how much the results actually move around.


In [ ]:
seeds = [1, 2, 3, 4, 5]

records = []
for model_name, use_memory in [("A: No memory", False), ("B: Memory + feedback", True)]:
    for seed in seeds:
        popularity = run_simulation(use_memory=use_memory, seed=seed)
        stats = summarize(popularity, model_name)
        stats["seed"] = seed
        records.append(stats)

repeated_runs = pd.DataFrame(records)
repeated_runs[["model", "seed", "top-5% share", "traffic entropy"]]


In [ ]:
reproducibility_summary = repeated_runs.groupby("model")[["top-5% share", "traffic entropy"]].agg(
    ["mean", "std", "min", "max"]
)
reproducibility_summary


**What to read off this table:** compare the `std` (standard deviation) columns between the two models. Model A's numbers barely move between runs — a small, tight `std`. Model B's numbers move noticeably more, run to run, even though nothing about the rules or the setup changed — only which random walker happened to be where, early on.

That's the real signature of "organized complexity" in Weaver's sense: not that the group behaves in a more complicated way, but that the group-level *outcome itself* stops being reliably predictable from the parameters alone, because the system has memory. No amount of knowing `n_walkers`, `follow_probability`, or the building layout in advance tells you exactly how concentrated Model B's traffic will end up being — you have to run it and see.


## Recap

| | Model A: No memory | Model B: Memory + feedback |
|---|---|---|
| what a walker knows about its goal | exact location, always | exact location, always |
| what a walker knows about the past | nothing | how popular each nearby cell has been |
| the one line of code that differs | random choice among candidates | usually the most popular candidate |
| result, single run | less concentrated traffic | more concentrated traffic |
| result, across 5 runs | nearly identical every time | noticeably different every time |

Same world. Same walkers. Same destinations. Same allowed moves. One tie-breaking rule, and everything downstream of it changes character, not just degree.
